# Advertisement line scores

Scores each transcript line with the classifier trained by [`hot_topic/train_classifier.py`](../hot_topic/train_classifier.py). The classifier's `noise` probability is written as `advertisement`.

Set `B` (Before) and `A` (After) in the configuration cell to include neighboring transcript lines as context. Both default to 0, so each line is classified by itself.

Input transcripts are the resegmented episode CSVs (`start`, `end`, `text`). Each episode is written to its own CSV under `out/ad_line_scores/`, with the same relative path as the input. `start`, `end`, and `text` are copied through so a scored file can be plotted against time.

Run the scoring cells first, then the plotting cells. Point `INPUT_ROOT` at the transcript directory before scoring.

In [1]:
import csv
import os
from itertools import batched
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

SCORE_COLUMN = "advertisement"

# INPUT_ROOT = Path("out/resegmented")
INPUT_ROOT = Path("out/whisper_segmented")
OUTPUT_ROOT = Path("out/ad_line_scores")
MODEL_PATH = Path("out/noise_classifier")

B = 0  # preceding lines to include
A = 0  # succeeding lines to include
BATCH_SIZE = 256
BUFFER_SIZE = 32  # GPU batches to hold before copying scores back to the CPU
DIGITS = 6
N_EPISODES = 100  # set to an int to score only the N largest transcripts
EPISODE_FILES = None  # or a list of Paths under INPUT_ROOT

SCORE_FORMAT = "{: ." + str(DIGITS) + "f}".replace(" ", "")

/blue/bonniejdorr/ethanlmines/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Score every episode

The trained classifier's `noise` probability is written as `advertisement`. Each target line can include `B` preceding and `A` succeeding lines as context. Lines are sorted longest-first before batching so padding stays small, then restored to transcript order. Episodes are processed largest-file-first.

In [2]:
def load_classifier(model_path):
    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()

    noise_index = None
    for index, label in getattr(model.config, "id2label", {}).items():
        if str(label).lower() == "noise":
            noise_index = int(index)
            break
    if noise_index is None:
        raise ValueError("Could not identify the noise label in the classifier")

    device = next(model.parameters()).device
    return tokenizer, model, noise_index, device


def episode_paths(root, files, n):
    root = root.resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Transcript directory does not exist: {root}")
    if files:
        paths = []
        for path in files:
            resolved = Path(path).resolve()
            if not resolved.is_relative_to(root):
                raise ValueError(f"input file is outside INPUT_ROOT: {path}")
            paths.append(resolved)
    else:
        paths = list(root.glob("**/*.csv"))
    paths = sorted(paths, key=os.path.getsize, reverse=True)
    if n is not None:
        paths = paths[:n]
    return paths


def score_lines(lines, tokenizer, model, noise_index, device, before=B, after=A):
    """Return the classifier's noise probability for each target line."""
    if before < 0 or after < 0:
        raise ValueError("before and after must be non-negative")
    if not lines:
        return []

    contextual_lines = [
        "\n".join(lines[max(0, index - before):index + after + 1])
        for index in range(len(lines))
    ]
    tokenized_samples = [
        tokenizer(line, truncation=True, max_length=256)
        for line in contextual_lines
    ]

    seq_lens = np.array([len(sample["input_ids"]) for sample in tokenized_samples])
    sort_inds = np.flip(np.argsort(seq_lens)).tolist()
    tokenized_samples = [tokenized_samples[i] for i in sort_inds]

    collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")
    gpu_batches = []
    for samples in batched(tokenized_samples, BATCH_SIZE):
        encoded = collator(list(samples))
        gpu_batches.append({key: value.to(device) for key, value in encoded.items()})
    tokenized_samples = None

    scores = []
    buffer = []
    with torch.inference_mode():
        for batch in tqdm(gpu_batches, desc="Scoring batches", leave=False):
            logits = model(**batch).logits
            batch_scores = torch.softmax(logits, dim=-1)[:, noise_index]
            buffer.append(batch_scores.detach())
            if len(buffer) >= BUFFER_SIZE:
                scores.extend(torch.cat(buffer).tolist())
                buffer = []
        if buffer:
            scores.extend(torch.cat(buffer).tolist())

    scores = [
        score
        for _, score in sorted(enumerate(scores), key=lambda i_score: sort_inds[i_score[0]])
    ]
    return [SCORE_FORMAT.format(score) for score in scores]


def score_episode(input_path, output_path, tokenizer, model, noise_index, device,
                  before=B, after=A):
    with open(input_path, newline="", encoding="utf-8") as source:
        rows = list(csv.DictReader(source))
    missing = {"start", "end", "text"} - set(rows[0] if rows else {"start", "end", "text"})
    if rows and missing:
        raise ValueError(f"{input_path} is missing columns: {sorted(missing)}")

    scores = score_lines(
        [row["text"] for row in rows],
        tokenizer,
        model,
        noise_index,
        device,
        before,
        after,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as destination:
        writer = csv.writer(destination)
        writer.writerow(["start", "end", "text", SCORE_COLUMN])
        writer.writerows(
            [row["start"], row["end"], row["text"], score]
            for row, score in zip(rows, scores)
        )
    return output_path


def score_transcripts(input_root=INPUT_ROOT, output_root=OUTPUT_ROOT, files=EPISODE_FILES,
                      n=N_EPISODES, before=B, after=A):
    tokenizer, model, noise_index, device = load_classifier(MODEL_PATH)
    written = []
    for input_path in tqdm(episode_paths(input_root, files, n), desc="Processing episodes"):
        relative_path = input_path.relative_to(input_root.resolve())
        output_path = Path(output_root) / relative_path
        written.append(
            score_episode(
                input_path, output_path, tokenizer, model, noise_index, device, before, after
            )
        )
    return written

In [3]:
scored_paths = score_transcripts()
print(f"Wrote {len(scored_paths)} episode CSVs under {OUTPUT_ROOT}")

Processing episodes: 100%|██████████| 100/100 [03:33<00:00,  2.13s/it]

Wrote 100 episode CSVs under out/ad_line_scores


## Plot one scored transcript

`plot_ad_scores` reads a CSV written above and plots `advertisement` against time. Pass any scored episode path.

Timestamps are interpreted automatically:

- values containing `:` are clock times (`HH:MM:SS`)
- otherwise the typical gap between line starts decides the unit (seconds, centiseconds, or milliseconds)

Pass `timestamp_unit="s"`, `"cs"`, or `"ms"` if the inferred unit is wrong. `smooth_lines` is a centered moving average, in lines rather than minutes, so a noisy per-line score still shows the stretches that look like ads.

In [4]:
def _clock_to_seconds(text):
    total = 0.0
    for part in str(text).strip().split(":"):
        total = total * 60 + float(part)
    return total


def timestamps_to_seconds(values, timestamp_unit="auto"):
    """Return seconds from episode start, and the unit name that was applied."""
    texts = [str(value).strip() for value in values]
    if timestamp_unit == "auto" and any(":" in text for text in texts):
        return [_clock_to_seconds(text) for text in texts], "clock"
    if timestamp_unit == "clock":
        return [_clock_to_seconds(text) for text in texts], "clock"

    numbers = [float(text) for text in texts]
    divisors = {"s": 1.0, "cs": 100.0, "ms": 1000.0}
    if timestamp_unit in divisors:
        divisor = divisors[timestamp_unit]
        unit = {"s": "seconds", "cs": "centiseconds", "ms": "milliseconds"}[timestamp_unit]
        return [number / divisor for number in numbers], unit
    if timestamp_unit != "auto":
        raise ValueError("timestamp_unit must be 'auto', 'clock', 's', 'cs', or 'ms'")

    gaps = [b - a for a, b in zip(numbers, numbers[1:]) if b > a]
    median_gap = float(np.median(gaps)) if gaps else 0.0
    # A spoken line is usually a few seconds. Millisecond gaps sit in the
    # thousands; whisper's native 10ms units sit in the hundreds.
    if median_gap >= 500:
        return [number / 1000.0 for number in numbers], "milliseconds"
    if median_gap >= 50:
        return [number / 100.0 for number in numbers], "centiseconds"
    return numbers, "seconds"


def rolling_mean(values, window):
    arr = np.asarray(values, dtype=float)
    if window <= 1 or arr.size == 0:
        return arr
    window = min(int(window), arr.size)
    pad_left = window // 2
    pad_right = window - 1 - pad_left
    padded = np.pad(arr, (pad_left, pad_right), mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(padded, kernel, mode="valid")


def plot_ad_scores(scored_csv, smooth_lines=25, timestamp_unit="auto", threshold=0.8):
    """Plot advertisement entailment scores for one scored episode CSV."""
    scored_csv = Path(scored_csv)
    with open(scored_csv, newline="", encoding="utf-8") as source:
        rows = list(csv.DictReader(source))
    if not rows:
        raise ValueError(f"{scored_csv} has no lines")
    required = {"start", "text", SCORE_COLUMN}
    missing = required - set(rows[0])
    if missing:
        raise ValueError(f"{scored_csv} is missing columns: {sorted(missing)}")

    seconds, unit = timestamps_to_seconds([row["start"] for row in rows], timestamp_unit)
    minutes = np.asarray(seconds, dtype=float) / 60.0
    scores = np.array([float(row[SCORE_COLUMN]) for row in rows])
    hover = [row["text"].strip().replace("\n", " ")[:240] for row in rows]

    order = np.argsort(minutes, kind="stable")
    minutes = minutes[order]
    scores = scores[order]
    hover = [hover[i] for i in order]
    smooth = rolling_mean(scores, smooth_lines)

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=minutes,
            y=scores,
            mode="markers",
            name="Line score",
            text=hover,
            hovertemplate="%{x:.2f} min<br>score=%{y:.3f}<br>%{text}<extra></extra>",
            marker=dict(size=6, color="rgba(120,120,120,0.45)"),
        )
    )
    if smooth_lines > 1:
        figure.add_trace(
            go.Scatter(
                x=minutes,
                y=smooth,
                mode="lines",
                name=f"{smooth_lines}-line mean",
                line=dict(width=2, color="#1f77b4"),
                hovertemplate="%{x:.2f} min<br>mean=%{y:.3f}<extra></extra>",
            )
        )
    if threshold is not None:
        figure.add_hline(
            y=threshold,
            line_dash="dash",
            line_color="#d62728",
            annotation_text=f"{threshold:g}",
            annotation_position="top left",
        )

    figure.update_layout(
        title=f"{scored_csv.name} ({unit})",
        xaxis_title="Minutes into episode",
        yaxis_title="P(This is an advertisement.)",
        yaxis=dict(range=[0, 1]),
        template="plotly_white",
        height=480,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    )
    print(f"{scored_csv}: {len(rows)} lines, timestamps read as {unit}")
    # figure.show()
    return figure

In [9]:
# scored = sorted(Path(OUTPUT_ROOT).glob("**/*.csv"))
output_path = Path(OUTPUT_ROOT) / '1328409/1328409_00014.mp3.csv'
# output_path = Path(OUTPUT_ROOT) / '49870/49870_00217.mp3.csv'
# output_path = Path(OUTPUT_ROOT) / '1017776/1017776_00002.mp3.csv'
# output_path = Path(OUTPUT_ROOT) / '5951054/5951054_00018.mp3.csv'
# plot_ad_scores(output_path, smooth_lines=25)
plot_ad_scores(output_path, smooth_lines=0)

out/ad_line_scores/1328409/1328409_00014.mp3.csv: 3572 lines, timestamps read as milliseconds
